# Problem Statement

## Business Context

A growing retail bank operating across several Asian markets is facing an increase in customer churn, with customers closing accounts or moving to competitors. This is affecting revenue, customer loyalty, and long-term growth.

The bank wants to use customer data to better understand the reasons behind churn and identify customers who are likely to leave. The key challenges are understanding varied customer behavior and moving from reactive retention efforts to a more proactive, data-driven approach.

## Objective

To overcome the limitations of traditional machine learning workflows, such as manual execution of data preparation, model training, testing, versioning, and deployment, the organization has hired you as a data scientist to implement a robust MLOps pipeline using GitHub Actions on Hugging Face. The objective is to build an automated and reproducible MLOps pipeline that streamlines the entire ML lifecycle - from code integration to model deployment - ensuring faster, more reliable access to the churn prediction model for geographically distributed teams, and enabling proactive, data-driven customer retention strategies.


# Prerequisites

* Create a GitHub repo
    - Go to ***GitHub Profile***
    - Click on ***Your repositories*** then select ***New***
      - Repository Name: ***MLOps***
      - Check the box ***README.md*** file
      - Click on ***Create repository***

* Adding Hugging Face space secrets to GitHub Actions to execute the workflow
  1. Go to Hugging Face ***Profile***
  2. Navigate to ***Access Token***
  3. Create a ***New token***
      - Token type ***Write***
      - Token Name ***MLOps***
      - Click on ***Create Token***
      - Copy the generated Token
  4. Now, go to GitHub repo
      - Click on ***Settings***
      - Navigate to ***Secrets and Variables***
      - Click on ***Actions***
      - Add a ***Repository secerts***
        - Name ***HF_TOKEN***
        - Secret: ***Paste the token created from the hugging face access tokens***
        - Click on ***Add secret***

* Create a Hugging Face space
    - Go to **Hugging Face**
    - Open your **Profile**
    - Click on **New Space**
      - Under the space creation, enter the below details
        - Space name: **Bank-Customer-Churn**
    (If you were trying with different names, be cautious when using a underscore `_` in space names, such as `frontend_space`, as it can cause exceptions when accessing the API URL. Always use an hyphen `-` instead, like `frontend-space`.)
        - Select the space SDK: **Docker**
        - Choose a Docker template: **Streamlit**
        - Click on **Create Space**

In [1]:
pwd

'c:\\Users\\J P Agarwal\\Desktop\\MLOps\\project-3 With MFLOW and CD\\mlops'

In [2]:
# Create a master folder to keep all files created when executing the below code cells
import os
os.makedirs("mlops", exist_ok=True)

# Model Building

## Data Registration

In [3]:
os.makedirs("mlops/data", exist_ok=True)

Once the **data** folder created after executing the above cell, please upload the **bank_customer_churn.csv** in to the folder

In [4]:
# Create a folder for storing the model building files
os.makedirs("mlops/model_building", exist_ok=True)

In [5]:
%%writefile mlops/model_building/data_register.py
from huggingface_hub.utils import RepositoryNotFoundError, HfHubHTTPError
from huggingface_hub import HfApi, create_repo
import os


repo_id = "krish21may/Bank-Customer-Churn-4"
repo_type = "dataset"

# Initialize API client
api = HfApi(token=os.getenv("HF_TOKEN"))

# Step 1: Check if the space exists
try:
    api.repo_info(repo_id=repo_id, repo_type=repo_type)
    print(f"Space '{repo_id}' already exists. Using it.")
except RepositoryNotFoundError:
    print(f"Space '{repo_id}' not found. Creating new space...")
    create_repo(repo_id=repo_id, repo_type=repo_type, private=False)
    print(f"Space '{repo_id}' created.")

api.upload_folder(
    folder_path="mlops/data",
    repo_id=repo_id,
    repo_type=repo_type,
)

Writing mlops/model_building/data_register.py


## Data Preparation

In [6]:
%%writefile mlops/model_building/prep.py
# for data manipulation
import pandas as pd
import sklearn
# for creating a folder
import os
# for data preprocessing and pipeline creation
from sklearn.model_selection import train_test_split
# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi

# Define constants for the dataset and output paths
api = HfApi(token=os.getenv("HF_TOKEN"))
DATASET_PATH = "hf://datasets/krish21may/Bank-Customer-Churn-4/bank_customer_churn.csv"
bank_dataset = pd.read_csv(DATASET_PATH)
print("Dataset loaded successfully.")

# Define the target variable for the classification task
target = 'Exited'

# List of numerical features in the dataset
numeric_features = [
    'CreditScore',       # Customer's credit score
    'Age',               # Customer's age
    'Tenure',            # Number of years the customer has been with the bank
    'Balance',           # Customer’s account balance
    'NumOfProducts',     # Number of products the customer has with the bank
    'HasCrCard',         # Whether the customer has a credit card (binary: 0 or 1)
    'IsActiveMember',    # Whether the customer is an active member (binary: 0 or 1)
    'EstimatedSalary'    # Customer’s estimated salary
]

# List of categorical features in the dataset
categorical_features = [
    'Geography',         # Country where the customer resides
]

# Define predictor matrix (X) using selected numeric and categorical features
X = bank_dataset[numeric_features + categorical_features]

# Define target variable
y = bank_dataset[target]


# Split dataset into train and test
# Split the dataset into training and test sets
Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y,              # Predictors (X) and target variable (y)
    test_size=0.2,     # 20% of the data is reserved for testing
    random_state=42    # Ensures reproducibility by setting a fixed random seed
)

Xtrain.to_csv("Xtrain.csv",index=False)
Xtest.to_csv("Xtest.csv",index=False)
ytrain.to_csv("ytrain.csv",index=False)
ytest.to_csv("ytest.csv",index=False)


files = ["Xtrain.csv","Xtest.csv","ytrain.csv","ytest.csv"]

for file_path in files:
    api.upload_file(
        path_or_fileobj=file_path,
        path_in_repo=file_path.split("/")[-1],  # just the filename
        repo_id="krish21may/Bank-Customer-Churn-4",
        repo_type="dataset",
    )

Writing mlops/model_building/prep.py


## Model Training

### Experimentation and Tracking (Development Environment)

In [ ]:
# JP - I ran only the ngrok part in git bash

#!pip install mlflow==3.0.1 pyngrok==7.2.12 -q

To get the ngrok authorization token, please go to this [link](https://dashboard.ngrok.com/authtokens), generate a new token, copy it, and paste it in the designated code line below.

In [10]:
import subprocess
import sys
import time
import webbrowser

# Start MLflow Server
process = subprocess.Popen([
    sys.executable,
    "-m",
    "mlflow",
    "server",
    "--host",
    "127.0.0.1",
    "--port",
    "5000"
])

# Wait for the server to start
time.sleep(5)

# MLflow Local URL
mlflow_url = "http://127.0.0.1:5000"

print("=" * 60)
print("MLflow UI is available at:")
print(mlflow_url)
print("=" * 60)

# Open automatically in the default browser
webbrowser.open(mlflow_url)

MLflow UI is available at:
http://127.0.0.1:5000


True

In [12]:
# Set the tracking URL for MLflow
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("MLOps_experiment10")

2026/07/12 03:09:33 INFO mlflow.tracking.fluent: Experiment with name 'MLOps_experiment10' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/905550251645901631', creation_time=1783805973118, experiment_id='905550251645901631', last_update_time=1783805973118, lifecycle_stage='active', name='MLOps_experiment10', tags={}>

In [14]:
import pandas as pd
import sklearn
# for creating a folder
import os
# for data preprocessing and pipeline creation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
# for model training, tuning, and evaluation
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, recall_score
# for model serialization
import joblib


bank_dataset = pd.read_csv("mlops/data/bank_customer_churn.csv")
print("Dataset loaded successfully.")

# Define the target variable for the classification task
target = 'Exited'

# List of numerical features in the dataset
numeric_features = [
    'CreditScore',       # Customer's credit score
    'Age',               # Customer's age
    'Tenure',            # Number of years the customer has been with the bank
    'Balance',           # Customer’s account balance
    'NumOfProducts',     # Number of products the customer has with the bank
    'HasCrCard',         # Whether the customer has a credit card (binary: 0 or 1)
    'IsActiveMember',    # Whether the customer is an active member (binary: 0 or 1)
    'EstimatedSalary'    # Customer’s estimated salary
]

# List of categorical features in the dataset
categorical_features = [
    'Geography',         # Country where the customer resides
]

# Define predictor matrix (X) using selected numeric and categorical features
X = bank_dataset[numeric_features + categorical_features]

# Define target variable
y = bank_dataset[target]


# Split dataset into train and test
# Split the dataset into training and test sets
Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y,              # Predictors (X) and target variable (y)
    test_size=0.2,     # 20% of the data is reserved for testing
    random_state=42    # Ensures reproducibility by setting a fixed random seed
)

# Set the clas weight to handle class imbalance
class_weight = ytrain.value_counts()[0] / ytrain.value_counts()[1]
class_weight

# Define the preprocessing steps
preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

# Define base XGBoost model
xgb_model = xgb.XGBClassifier(scale_pos_weight=class_weight, random_state=42)


# Define hyperparameter grid
param_grid = {
    'xgbclassifier__n_estimators': [50, 75, 100],    # number of tree to build
    'xgbclassifier__max_depth': [2, 3],    # maximum depth of each tree
    'xgbclassifier__colsample_bytree': [0.4, 0.6],    # percentage of attributes to be considered (randomly) for each tree
    'xgbclassifier__colsample_bylevel': [0.4, 0.6],    # percentage of attributes to be considered (randomly) for each level of a tree
    'xgbclassifier__learning_rate': [0.01, 0.1],    # learning rate
    'xgbclassifier__reg_lambda': [0.4, 0.6],    # L2 regularization factor
}

# Model pipeline
model_pipeline = make_pipeline(preprocessor, xgb_model)

with mlflow.start_run():
    # Hyperparameter tuning
    grid_search = GridSearchCV(model_pipeline, param_grid, cv=5, n_jobs=-1)
    grid_search.fit(Xtrain, ytrain)

    # Log all parameter combinations and their mean test scores
    results = grid_search.cv_results_
    for i in range(len(results['params'])):
        param_set = results['params'][i]
        mean_score = results['mean_test_score'][i]
        std_score = results['std_test_score'][i]

        # Log each combination as a separate MLflow run
        with mlflow.start_run(nested=True):
            mlflow.log_params(param_set)
            mlflow.log_metric("mean_test_score", mean_score)
            mlflow.log_metric("std_test_score", std_score)

    # Log best parameters separately in main run
    mlflow.log_params(grid_search.best_params_)

    # Store and evaluate the best model
    best_model = grid_search.best_estimator_

    classification_threshold = 0.45

    y_pred_train_proba = best_model.predict_proba(Xtrain)[:, 1]
    y_pred_train = (y_pred_train_proba >= classification_threshold).astype(int)

    y_pred_test_proba = best_model.predict_proba(Xtest)[:, 1]
    y_pred_test = (y_pred_test_proba >= classification_threshold).astype(int)

    train_report = classification_report(ytrain, y_pred_train, output_dict=True)
    test_report = classification_report(ytest, y_pred_test, output_dict=True)

    mlflow.log_metrics({
        "train_accuracy": train_report['accuracy'],
        "train_precision": train_report['1']['precision'],
        "train_recall": train_report['1']['recall'],
        "train_f1-score": train_report['1']['f1-score'],
        "test_accuracy": test_report['accuracy'],
        "test_precision": test_report['1']['precision'],
        "test_recall": test_report['1']['recall'],
        "test_f1-score": test_report['1']['f1-score']
    })

Dataset loaded successfully.
🏃 View run invincible-toad-931 at: http://127.0.0.1:5000/#/experiments/905550251645901631/runs/e3f365fd93ee473fa54fce80fc5296df
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/905550251645901631
🏃 View run flawless-ant-217 at: http://127.0.0.1:5000/#/experiments/905550251645901631/runs/9fcb626eacf8499e981f19bb7be87b9d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/905550251645901631
🏃 View run thoughtful-ox-305 at: http://127.0.0.1:5000/#/experiments/905550251645901631/runs/7fc096ec35db4a85885ea6ec6c0be137
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/905550251645901631
🏃 View run debonair-tern-795 at: http://127.0.0.1:5000/#/experiments/905550251645901631/runs/14eb2469ca0045fc89e720ee261d40bb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/905550251645901631
🏃 View run dazzling-shrike-351 at: http://127.0.0.1:5000/#/experiments/905550251645901631/runs/189e7ccbe1db422cac0e4f6a5b735ce4
🧪 View experiment at: http://1

- As we can see, all the experiments conducted during hyperparameter tuning are being logged by MLflow.
- Upon clicking the links in the output of the above code cell, we can check the tracking on MLflow.

Now that we've tested the experimentation tracking with MLflow in a development environment, let's convert this to the required script for production environment usage.

### Experimentation and Tracking (Production Environment)

In [15]:
%%writefile mlops/model_building/train.py
# for data manipulation
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
# for model training, tuning, and evaluation
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, recall_score
# for model serialization
import joblib
# for creating a folder
import os
# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError, HfHubHTTPError
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mlops-training-experiment")

api = HfApi()


Xtrain_path = "hf://datasets/krish21may/Bank-Customer-Churn-4/Xtrain.csv"
Xtest_path = "hf://datasets/krish21may/Bank-Customer-Churn-4/Xtest.csv"
ytrain_path = "hf://datasets/krish21may/Bank-Customer-Churn-4/ytrain.csv"
ytest_path = "hf://datasets/krish21may/Bank-Customer-Churn-4/ytest.csv"

Xtrain = pd.read_csv(Xtrain_path)
Xtest = pd.read_csv(Xtest_path)
ytrain = pd.read_csv(ytrain_path)
ytest = pd.read_csv(ytest_path)


# List of numerical features in the dataset
numeric_features = [
    'CreditScore',       # Customer's credit score
    'Age',               # Customer's age
    'Tenure',            # Number of years the customer has been with the bank
    'Balance',           # Customer’s account balance
    'NumOfProducts',     # Number of products the customer has with the bank
    'HasCrCard',         # Whether the customer has a credit card (binary: 0 or 1)
    'IsActiveMember',    # Whether the customer is an active member (binary: 0 or 1)
    'EstimatedSalary'    # Customer’s estimated salary
]

# List of categorical features in the dataset
categorical_features = [
    'Geography',         # Country where the customer resides
]


# Set the clas weight to handle class imbalance
class_weight = ytrain.value_counts()[0] / ytrain.value_counts()[1]
class_weight

# Define the preprocessing steps
preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

# Define base XGBoost model
xgb_model = xgb.XGBClassifier(scale_pos_weight=class_weight, random_state=42)

# Define hyperparameter grid
param_grid = {
    'xgbclassifier__n_estimators': [50, 75, 100, 125, 150],    # number of tree to build
    'xgbclassifier__max_depth': [2, 3, 4],    # maximum depth of each tree
    'xgbclassifier__colsample_bytree': [0.4, 0.5, 0.6],    # percentage of attributes to be considered (randomly) for each tree
    'xgbclassifier__colsample_bylevel': [0.4, 0.5, 0.6],    # percentage of attributes to be considered (randomly) for each level of a tree
    'xgbclassifier__learning_rate': [0.01, 0.05, 0.1],    # learning rate
    'xgbclassifier__reg_lambda': [0.4, 0.5, 0.6],    # L2 regularization factor
}

# Model pipeline
model_pipeline = make_pipeline(preprocessor, xgb_model)

# Start MLflow run
with mlflow.start_run():
    # Hyperparameter tuning
    grid_search = GridSearchCV(model_pipeline, param_grid, cv=5, n_jobs=-1)
    grid_search.fit(Xtrain, ytrain)

    # Log all parameter combinations and their mean test scores
    results = grid_search.cv_results_
    for i in range(len(results['params'])):
        param_set = results['params'][i]
        mean_score = results['mean_test_score'][i]
        std_score = results['std_test_score'][i]

        # Log each combination as a separate MLflow run
        with mlflow.start_run(nested=True):
            mlflow.log_params(param_set)
            mlflow.log_metric("mean_test_score", mean_score)
            mlflow.log_metric("std_test_score", std_score)

    # Log best parameters separately in main run
    mlflow.log_params(grid_search.best_params_)

    # Store and evaluate the best model
    best_model = grid_search.best_estimator_

    classification_threshold = 0.45

    y_pred_train_proba = best_model.predict_proba(Xtrain)[:, 1]
    y_pred_train = (y_pred_train_proba >= classification_threshold).astype(int)

    y_pred_test_proba = best_model.predict_proba(Xtest)[:, 1]
    y_pred_test = (y_pred_test_proba >= classification_threshold).astype(int)

    train_report = classification_report(ytrain, y_pred_train, output_dict=True)
    test_report = classification_report(ytest, y_pred_test, output_dict=True)

    # Log the metrics for the best model
    mlflow.log_metrics({
        "train_accuracy": train_report['accuracy'],
        "train_precision": train_report['1']['precision'],
        "train_recall": train_report['1']['recall'],
        "train_f1-score": train_report['1']['f1-score'],
        "test_accuracy": test_report['accuracy'],
        "test_precision": test_report['1']['precision'],
        "test_recall": test_report['1']['recall'],
        "test_f1-score": test_report['1']['f1-score']
    })

    # Save the model locally
    model_path = "best_churn_model_v1.joblib"
    joblib.dump(best_model, model_path)

    # Log the model artifact
    mlflow.log_artifact(model_path, artifact_path="model")
    print(f"Model saved as artifact at: {model_path}")

    # Upload to Hugging Face
    repo_id = "krish21may/Bank-Customer-Churn-4"
    repo_type = "model"

    # Step 1: Check if the space exists
    try:
        api.repo_info(repo_id=repo_id, repo_type=repo_type)
        print(f"Space '{repo_id}' already exists. Using it.")
    except RepositoryNotFoundError:
        print(f"Space '{repo_id}' not found. Creating new space...")
        create_repo(repo_id=repo_id, repo_type=repo_type, private=False)
        print(f"Space '{repo_id}' created.")

    # create_repo("churn-model", repo_type="model", private=False)
    api.upload_file(
        path_or_fileobj="best_churn_model_v1.joblib",
        path_in_repo="best_churn_model_v1.joblib",
        repo_id=repo_id,
        repo_type=repo_type,
    )

Writing mlops/model_building/train.py


# Deployment

## Dockerfile

In [16]:
os.makedirs("mlops/deployment", exist_ok=True)

In [17]:
%%writefile mlops/deployment/Dockerfile
# Use a minimal base image with Python 3.9 installed
FROM python:3.9

# Set the working directory inside the container to /app
WORKDIR /app

# Copy all files from the current directory on the host to the container's /app directory
COPY . .

# Install Python dependencies listed in requirements.txt
RUN pip3 install -r requirements.txt

RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user \
	PATH=/home/user/.local/bin:$PATH

WORKDIR $HOME/app

COPY --chown=user . $HOME/app

# Define the command to run the Streamlit app on port "8501" and make it accessible externally
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

Writing mlops/deployment/Dockerfile


## Streamlit App

In [23]:
%%writefile mlops/deployment/app.py
import streamlit as st
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download
import joblib
import io
import plotly.express as px
import plotly.graph_objects as go

# Set Page Config
st.set_page_config(
    page_title="Bank Customer Churn Intelligence",
    page_icon="🏦",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Sidebar Theme Selector & System Controls
with st.sidebar:
    st.image("https://img.icons8.com/isometric/100/bank.png", width=70)
    st.title("Control Panel")
    
    st.markdown("### 🎨 Theme Selector")
    theme_mode = st.radio(
        "Choose Theme:",
        ["🌙 Dark Mode", "☀️ Light Mode"],
        index=0
    )
    is_dark = "Dark" in theme_mode

    st.markdown("---")
    st.markdown("### ⚙️ System Metadata")
    st.info("""
    - **Model Architecture**: Tuned XGBoost
    - **Decision Threshold**: `0.45`
    - **Hugging Face Hub**: `krish21may/Bank-Customer-Churn-4`
    - **Engine**: Plotly Interactive Visuals
    """)

# Dynamic CSS Theme Tokens
if is_dark:
    bg_color = "#0F172A"
    card_bg = "#1E293B"
    text_primary = "#F8FAFC"
    text_secondary = "#94A3B8"
    border_color = "#334155"
    accent_color = "#38BDF8"
    plotly_template = "plotly_dark"
    card_shadow = "0 4px 12px rgba(0, 0, 0, 0.4)"
else:
    bg_color = "#F8FAFC"
    card_bg = "#FFFFFF"
    text_primary = "#0F172A"
    text_secondary = "#475569"
    border_color = "#E2E8F0"
    accent_color = "#2563EB"
    plotly_template = "plotly_white"
    card_shadow = "0 4px 6px -1px rgba(0, 0, 0, 0.05)"

st.markdown(f"""
<style>
    .stApp {{
        background-color: {bg_color};
        color: {text_primary};
    }}
    .main-header {{
        font-size: 2.2rem;
        font-weight: 800;
        color: {accent_color};
        margin-bottom: 0.2rem;
    }}
    .sub-header {{
        font-size: 1.05rem;
        color: {text_secondary};
        margin-bottom: 1.5rem;
    }}
    .custom-card {{
        background-color: {card_bg};
        border-radius: 12px;
        padding: 20px;
        border: 1px solid {border_color};
        box-shadow: {card_shadow};
        margin-bottom: 20px;
        color: {text_primary};
    }}
    .stButton>button {{
        background-color: {accent_color};
        color: #FFFFFF !important;
        font-size: 1.05rem;
        font-weight: 600;
        border-radius: 8px;
        padding: 0.65rem 2rem;
        width: 100%;
        border: none;
        transition: all 0.3s ease;
    }}
    .stButton>button:hover {{
        opacity: 0.9;
        transform: translateY(-1px);
    }}
</style>
""", unsafe_allow_html=True)

# Load Model with Caching
@st.cache_resource
def load_churn_model():
    try:
        model_path = hf_hub_download(
            repo_id="krish21may/Bank-Customer-Churn-4", 
            filename="best_churn_model.joblib"
        )
        return joblib.load(model_path)
    except Exception as e:
        st.error(f"Error loading model from Hugging Face Hub: {e}")
        return None

model = load_churn_model()

# Main Title Banner
st.markdown('<div class="main-header">🏦 Bank Customer Churn Intelligence & Visual Analytics</div>', unsafe_allow_html=True)
st.markdown('<div class="sub-header">AI-Powered Risk Assessment, Financial Modeling & Dynamic Interactive Dashboards</div>', unsafe_allow_html=True)

# Navigation Tabs
tab_single, tab_batch, tab_analytics = st.tabs([
    "👤 Single Customer Prediction", 
    "📁 Batch CSV Upload & Inference", 
    "📊 Portfolio Visual Analytics"
])

REQUIRED_COLUMNS = [
    'CreditScore', 'Geography', 'Age', 'Tenure', 
    'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary'
]

# ==============================================================================
# TAB 1: SINGLE CUSTOMER PREDICTION & GAUGE CHART
# ==============================================================================
with tab_single:
    st.markdown("### 👤 Single Customer Risk Profiler")
    
    # Preset Profiles
    c_preset, _ = st.columns([2, 1])
    with c_preset:
        preset = st.radio(
            "Load Sample Customer Scenario:",
            ["Custom Input", "⚠️ High Churn Risk Customer", "✅ Low Churn Risk Customer"],
            horizontal=True
        )
    
    if preset == "⚠️ High Churn Risk Customer":
        def_credit, def_geo, def_age, def_tenure = 590, "Germany", 52, 2
        def_balance, def_num_prod, def_card, def_active, def_salary = 125000.0, 1, "Yes", "No", 75000.0
    elif preset == "✅ Low Churn Risk Customer":
        def_credit, def_geo, def_age, def_tenure = 750, "France", 28, 7
        def_balance, def_num_prod, def_card, def_active, def_salary = 45000.0, 2, "Yes", "Yes", 95000.0
    else:
        def_credit, def_geo, def_age, def_tenure = 650, "France", 38, 5
        def_balance, def_num_prod, def_card, def_active, def_salary = 50000.0, 1, "Yes", "Yes", 60000.0

    col_input, col_results = st.columns([1.1, 1], gap="large")

    with col_input:
        st.markdown("##### 📋 Demographic & Financial Inputs")
        
        c1, c2 = st.columns(2)
        with c1:
            Age = st.number_input("Age (Years)", min_value=18, max_value=100, value=def_age, step=1, key="s_age")
            Geography = st.selectbox("Geography", ["France", "Germany", "Spain"], index=["France", "Germany", "Spain"].index(def_geo), key="s_geo")
            Tenure = st.number_input("Tenure (Years with Bank)", min_value=0, max_value=20, value=def_tenure, step=1, key="s_tenure")
            EstimatedSalary = st.number_input("Estimated Annual Salary ($)", min_value=0.0, max_value=500000.0, value=float(def_salary), step=1000.0, key="s_salary")

        with c2:
            CreditScore = st.number_input("Credit Score", min_value=300, max_value=900, value=def_credit, step=5, key="s_credit")
            Balance = st.number_input("Account Balance ($)", min_value=0.0, max_value=1000000.0, value=float(def_balance), step=5000.0, key="s_balance")
            NumOfProducts = st.slider("Number of Bank Products", min_value=1, max_value=4, value=def_num_prod, key="s_products")
            HasCrCard = st.selectbox("Has Credit Card?", ["Yes", "No"], index=0 if def_card == "Yes" else 1, key="s_card")
            IsActiveMember = st.selectbox("Is Active Member?", ["Yes", "No"], index=0 if def_active == "Yes" else 1, key="s_active")

        predict_btn = st.button("🔍 Predict Customer Churn Risk")

    single_input = pd.DataFrame([{
        'CreditScore': CreditScore,
        'Geography': Geography,
        'Age': Age,
        'Tenure': Tenure,
        'Balance': Balance,
        'NumOfProducts': NumOfProducts,
        'HasCrCard': 1 if HasCrCard == "Yes" else 0,
        'IsActiveMember': 1 if IsActiveMember == "Yes" else 0,
        'EstimatedSalary': EstimatedSalary
    }])

    with col_results:
        st.markdown("##### 📊 Real-Time Risk Intelligence")
        
        if predict_btn or preset != "Custom Input":
            if model is not None:
                prob = float(model.predict_proba(single_input)[0, 1])
                threshold = 0.45
                is_churn = prob >= threshold
                
                # Plotly Gauge Chart for Churn Risk
                fig_gauge = go.Figure(go.Indicator(
                    mode="gauge+number",
                    value=prob * 100,
                    number={'suffix': '%', 'font': {'size': 36, 'color': accent_color}},
                    title={'text': "Churn Risk Index", 'font': {'size': 18, 'color': text_primary}},
                    gauge={
                        'axis': {'range': [0, 100], 'tickwidth': 1, 'tickcolor': text_secondary},
                        'bar': {'color': "#EF4444" if is_churn else "#10B981"},
                        'bgcolor': card_bg,
                        'borderwidth': 1,
                        'bordercolor': border_color,
                        'steps': [
                            {'range': [0, 30], 'color': 'rgba(16, 185, 129, 0.2)'},
                            {'range': [30, 45], 'color': 'rgba(245, 158, 11, 0.2)'},
                            {'range': [45, 100], 'color': 'rgba(239, 68, 68, 0.2)'}
                        ],
                        'threshold': {
                            'line': {'color': "red", 'width': 3},
                            'thickness': 0.75,
                            'value': threshold * 100
                        }
                    }
                ))
                fig_gauge.update_layout(
                    template=plotly_template,
                    height=250,
                    margin=dict(l=20, r=20, t=40, b=20),
                    paper_bgcolor='rgba(0,0,0,0)',
                    plot_bgcolor='rgba(0,0,0,0)'
                )
                st.plotly_chart(fig_gauge, use_container_width=True)
                
                # Metrics Row
                m1, m2 = st.columns(2)
                with m1:
                    st.metric("Predicted Churn Probability", f"{prob * 100:.1f}%", f"{(prob - threshold) * 100:+.1f}% vs threshold", delta_color="inverse")
                with m2:
                    st.metric("Risk Status", "HIGH CHURN RISK" if is_churn else "LOW CHURN RISK", "Action Required" if is_churn else "Stable Customer", delta_color="inverse" if is_churn else "normal")

                if is_churn:
                    st.error(f"### ⚠️ High Risk Customer Flagged ({prob * 100:.1f}%)")
                    st.markdown("""
                    **Recommended Personal Retention Actions:**
                    - 📞 Schedule personal relationship check-in within 48 hours.
                    - 💰 Waive annual fees & offer promotional interest rate on deposits.
                    - 💳 Cross-sell relevant financial protection products.
                    """)
                else:
                    st.success(f"### ✅ Customer Status Stable ({prob * 100:.1f}%)")
                    st.markdown("""
                    **Recommended Engagement Strategy:**
                    - 💳 Offer premium rewards credit card upgrade.
                    - 📈 Invite customer to private wealth advisory seminar.
                    """)
            else:
                st.warning("Model is not loaded properly.")
        else:
            st.info("👈 Enter details on the left and click **'Predict Customer Churn Risk'**.")

# ==============================================================================
# TAB 2: BATCH CSV UPLOAD & BULK INFERENCE
# ==============================================================================
with tab_batch:
    st.markdown("### 📁 Batch Prediction Engine")
    st.markdown("Upload a CSV file of bank customers to generate bulk predictions and automated risk strategies.")
    
    c_up, c_template = st.columns([2, 1], gap="large")
    
    with c_template:
        st.markdown("##### 📥 CSV Template Generator")
        sample_df = pd.DataFrame([
            {'CreditScore': 619, 'Geography': 'France', 'Age': 42, 'Tenure': 2, 'Balance': 0.0, 'NumOfProducts': 1, 'HasCrCard': 1, 'IsActiveMember': 1, 'EstimatedSalary': 101348.88},
            {'CreditScore': 608, 'Geography': 'Spain', 'Age': 41, 'Tenure': 1, 'Balance': 83807.86, 'NumOfProducts': 1, 'HasCrCard': 0, 'IsActiveMember': 1, 'EstimatedSalary': 112542.58},
            {'CreditScore': 502, 'Geography': 'France', 'Age': 42, 'Tenure': 8, 'Balance': 159660.8, 'NumOfProducts': 3, 'HasCrCard': 1, 'IsActiveMember': 0, 'EstimatedSalary': 113931.57},
            {'CreditScore': 699, 'Geography': 'Germany', 'Age': 39, 'Tenure': 1, 'Balance': 120000.0, 'NumOfProducts': 2, 'HasCrCard': 1, 'IsActiveMember': 0, 'EstimatedSalary': 93826.63},
            {'CreditScore': 850, 'Geography': 'Spain', 'Age': 43, 'Tenure': 2, 'Balance': 125510.82, 'NumOfProducts': 1, 'HasCrCard': 1, 'IsActiveMember': 1, 'EstimatedSalary': 79084.1}
        ])
        buf = io.BytesIO()
        sample_df.to_csv(buf, index=False)
        st.download_button(
            "📄 Download Sample CSV Template",
            buf.getvalue(),
            "sample_bank_customers.csv",
            "text/csv"
        )
        
    with c_up:
        uploaded_file = st.file_uploader("Upload Customer CSV File", type=["csv"])

    if uploaded_file is not None:
        try:
            batch_df = pd.read_csv(uploaded_file)
            st.success(f"File **'{uploaded_file.name}'** loaded ({len(batch_df)} records).")
            
            missing_cols = [c for c in REQUIRED_COLUMNS if c not in batch_df.columns]
            if missing_cols:
                st.error(f"Missing required columns: `{missing_cols}`")
            else:
                proc_df = batch_df[REQUIRED_COLUMNS].copy()
                for b_col in ['HasCrCard', 'IsActiveMember']:
                    if proc_df[b_col].dtype == object:
                        proc_df[b_col] = proc_df[b_col].apply(lambda x: 1 if str(x).strip().lower() in ['1', 'yes', 'true'] else 0)
                
                if model is not None:
                    probs = model.predict_proba(proc_df)[:, 1]
                    threshold = 0.45
                    preds = (probs >= threshold).astype(int)
                    
                    res_df = batch_df.copy()
                    res_df['Churn_Probability_%'] = np.round(probs * 100, 2)
                    res_df['Risk_Status'] = np.where(preds == 1, 'HIGH RISK ⚠️', 'LOW RISK ✅')
                    res_df['Strategy'] = np.where(preds == 1, 'RM Check-in & Rate Waiver', 'Upsell Wealth/Credit')
                    
                    # Store in session state for analytics tab
                    st.session_state['batch_results'] = res_df
                    
                    # Batch Metrics
                    st.markdown("---")
                    st.markdown("##### 📊 Batch Summary KPI Indicators")
                    k1, k2, k3, k4 = st.columns(4)
                    k1.metric("Total Records", len(res_df))
                    k2.metric("High Risk Count", int(np.sum(preds == 1)), f"{np.mean(preds)*100:.1f}% risk rate", delta_color="inverse")
                    k3.metric("Low Risk Count", int(np.sum(preds == 0)))
                    k4.metric("Avg Churn Prob", f"{np.mean(probs)*100:.1f}%")

                    # Donut Chart for Batch Overview
                    st.markdown("---")
                    c_chart1, c_chart2 = st.columns(2)
                    with c_chart1:
                        st.markdown("##### 🎯 Risk Classification Breakdown")
                        fig_donut = px.pie(
                            res_df, 
                            names='Risk_Status', 
                            title="Portfolio Risk Distribution",
                            hole=0.4,
                            color='Risk_Status',
                            color_discrete_map={'HIGH RISK ⚠️': '#EF4444', 'LOW RISK ✅': '#10B981'},
                            template=plotly_template
                        )
                        fig_donut.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
                        st.plotly_chart(fig_donut, use_container_width=True)
                    
                    with c_chart2:
                        st.markdown("##### 🌍 Churn Risk by Geography")
                        fig_geo = px.histogram(
                            res_df,
                            x='Geography',
                            color='Risk_Status',
                            barmode='group',
                            title="Geographic Risk Comparison",
                            color_discrete_map={'HIGH RISK ⚠️': '#EF4444', 'LOW RISK ✅': '#10B981'},
                            template=plotly_template
                        )
                        fig_geo.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
                        st.plotly_chart(fig_geo, use_container_width=True)

                    # Table Display
                    st.markdown("##### 📋 Full Batch Predictions Table")
                    st.dataframe(res_df, use_container_width=True)
                    
                    # Download Predictions
                    out_buf = io.BytesIO()
                    res_df.to_csv(out_buf, index=False)
                    st.download_button(
                        "📥 Download Batch Predictions CSV",
                        out_buf.getvalue(),
                        f"churn_predictions_{uploaded_file.name}",
                        "text/csv"
                    )
        except Exception as err:
            st.error(f"Error parsing CSV: {err}")

# ==============================================================================
# TAB 3: PORTFOLIO VISUAL ANALYTICS (PLOTLY DASHBOARD)
# ==============================================================================
with tab_analytics:
    st.markdown("### 📊 Portfolio Visual Analytics Dashboard")
    st.markdown("Interactive visual discovery platform analyzing feature correlations, age clusters, and balance distributions.")
    
    if 'batch_results' in st.session_state and st.session_state['batch_results'] is not None:
        df_ana = st.session_state['batch_results']
    else:
        # Load sample dataset if no batch uploaded yet
        st.info("💡 Displaying baseline analytics dataset. Upload a batch CSV in Tab 2 to visualize your custom portfolio.")
        df_ana = pd.DataFrame([
            {'CreditScore': 619, 'Geography': 'France', 'Age': 42, 'Tenure': 2, 'Balance': 0.0, 'NumOfProducts': 1, 'HasCrCard': 1, 'IsActiveMember': 1, 'EstimatedSalary': 101348.88, 'Churn_Probability_%': 62.4, 'Risk_Status': 'HIGH RISK ⚠️'},
            {'CreditScore': 608, 'Geography': 'Spain', 'Age': 41, 'Tenure': 1, 'Balance': 83807.86, 'NumOfProducts': 1, 'HasCrCard': 0, 'IsActiveMember': 1, 'EstimatedSalary': 112542.58, 'Churn_Probability_%': 18.2, 'Risk_Status': 'LOW RISK ✅'},
            {'CreditScore': 502, 'Geography': 'France', 'Age': 42, 'Tenure': 8, 'Balance': 159660.8, 'NumOfProducts': 3, 'HasCrCard': 1, 'IsActiveMember': 0, 'EstimatedSalary': 113931.57, 'Churn_Probability_%': 74.8, 'Risk_Status': 'HIGH RISK ⚠️'},
            {'CreditScore': 699, 'Geography': 'Germany', 'Age': 55, 'Tenure': 1, 'Balance': 120000.0, 'NumOfProducts': 2, 'HasCrCard': 1, 'IsActiveMember': 0, 'EstimatedSalary': 93826.63, 'Churn_Probability_%': 81.5, 'Risk_Status': 'HIGH RISK ⚠️'},
            {'CreditScore': 850, 'Geography': 'Spain', 'Age': 29, 'Tenure': 7, 'Balance': 45000.0, 'NumOfProducts': 2, 'HasCrCard': 1, 'IsActiveMember': 1, 'EstimatedSalary': 79084.1, 'Churn_Probability_%': 12.1, 'Risk_Status': 'LOW RISK ✅'},
            {'CreditScore': 720, 'Geography': 'Germany', 'Age': 48, 'Tenure': 4, 'Balance': 110000.0, 'NumOfProducts': 1, 'HasCrCard': 1, 'IsActiveMember': 0, 'EstimatedSalary': 88000.0, 'Churn_Probability_%': 68.3, 'Risk_Status': 'HIGH RISK ⚠️'},
            {'CreditScore': 630, 'Geography': 'France', 'Age': 33, 'Tenure': 6, 'Balance': 30000.0, 'NumOfProducts': 2, 'HasCrCard': 1, 'IsActiveMember': 1, 'EstimatedSalary': 55000.0, 'Churn_Probability_%': 15.4, 'Risk_Status': 'LOW RISK ✅'}
        ])

    ca1, ca2 = st.columns(2)
    
    with ca1:
        st.markdown("##### 📈 Age vs. Account Balance vs. Risk Score")
        fig_scatter = px.scatter(
            df_ana,
            x='Age',
            y='Balance',
            size='Churn_Probability_%' if 'Churn_Probability_%' in df_ana.columns else None,
            color='Risk_Status' if 'Risk_Status' in df_ana.columns else 'Geography',
            hover_data=['CreditScore', 'Geography', 'NumOfProducts'],
            title="Customer Demographics & Account Balance Cluster",
            color_discrete_map={'HIGH RISK ⚠️': '#EF4444', 'LOW RISK ✅': '#10B981'},
            template=plotly_template
        )
        fig_scatter.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
        st.plotly_chart(fig_scatter, use_container_width=True)

    with ca2:
        st.markdown("##### 🛍️ Churn Distribution by Product Count")
        fig_prod = px.box(
            df_ana,
            x='NumOfProducts',
            y='Churn_Probability_%' if 'Churn_Probability_%' in df_ana.columns else 'Balance',
            color='Risk_Status' if 'Risk_Status' in df_ana.columns else None,
            title="Churn Probability Spread across Product Holdings",
            color_discrete_map={'HIGH RISK ⚠️': '#EF4444', 'LOW RISK ✅': '#10B981'},
            template=plotly_template
        )
        fig_prod.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
        st.plotly_chart(fig_prod, use_container_width=True)

# Footer
st.markdown("---")
st.markdown("<div style='text-align: center; color: #9CA3AF;'>Bank Customer Churn Intelligence & Analytics Engine | Streamlit • XGBoost • Plotly • Hugging Face</div>", unsafe_allow_html=True)


Overwriting mlops/deployment/app.py


## Dependency Handling

In [19]:
%%writefile mlops/deployment/requirements.txt
pandas==2.2.2
huggingface_hub==0.32.6
streamlit==1.43.2
joblib==1.5.1
scikit-learn==1.6.0
xgboost==2.1.4
mlflow==3.0.1

Writing mlops/deployment/requirements.txt


# Hosting

In [20]:
os.makedirs("mlops/hosting", exist_ok=True)

In [21]:
%%writefile mlops/hosting/hosting.py
from huggingface_hub import HfApi
import os

api = HfApi(token=os.getenv("HF_TOKEN"))
api.upload_folder(
    folder_path="mlops/deployment",     # the local folder containing your files
    repo_id="krish21may/Bank-Customer-Churn-4",          # the target repo
    repo_type="space",                      # dataset, model, or space
    path_in_repo="",                          # optional: subfolder path inside the repo
)

Writing mlops/hosting/hosting.py


# Create and Automate MLOps Pipeline with GitHub Action Workflows using CI/CD

## Actions Workflow YAML File

* A YAML file is a simple, human-readable file used to store configuration settings.
* YAML stands for Yet Another Markup Language or YAML Ain't Markup Language (a recursive acronym).
* It uses indentation (spaces) to show structure, like folders inside folders.
* Each line contains a key and a value, making it easy to organize data.
* YAML is often used in automation tools, cloud setups, and app settings.

Here's the YAML file we'd need for our use case.

```
name: MLOps pipeline

on:
  push:
    branches:
      - main  # Automatically triggers on push to the main branch
jobs:
  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r mlops/requirements.txt
      - name: Upload Dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python mlops/model_building/data_register.py

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r mlops/requirements.txt
      - name: Run Data Preparation
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python mlops/model_building/prep.py


  model-traning:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r mlops/requirements.txt
      - name: Start MLflow Server
        run: |
          nohup mlflow ui --host 0.0.0.0 --port 5000 &  # Run MLflow UI in the background
          sleep 5  # Wait for a moment to let the server start
      - name: Model Building
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python mlops/model_building/train.py

  deploy-hosting:
    runs-on: ubuntu-latest
    needs: [model-traning,data-prep,register-dataset]
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r mlops/requirements.txt
      - name: Push files to Frontend Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python mlops/hosting/hosting.py

```

**Note:** To use this YAML file for our use case, we need to

1. Go to the GitHub repository for the project
2. Create a folder named ***.github/workflows/***
3. In the above folder, create a file named ***pipeline.yml***
4. Copy and paste the above content for the YAML file into the ***pipeline.yml*** file

## Requirements file for the Github Actions Workflow

In [22]:
%%writefile mlops/requirements.txt
huggingface_hub==0.32.6
datasets==3.6.0
pandas==2.2.2
scikit-learn==1.6.0
xgboost==2.1.4
mlflow==3.0.1

Writing mlops/requirements.txt


## Github Authentication and Push Files

* Before moving forward, we need to generate a secret token to push files directly from Colab to the GitHub repository.
* Please follow the below instructions to create the GitHub token:
    - Open your GitHub profile.
    - Click on ***Settings***.
    - Go to ***Developer Settings***.
    - Expand the ***Personal access tokens*** section and select ***Tokens (classic)***.
    - Click ***Generate new token***, then choose ***Generate new token (classic)***.
    - Add a note and select all required scopes.
    - Click ***Generate token***.
    - Copy the generated token and store it safely in a notepad.